In [2]:

# Step 1: Install or update the UCI dataset package
%pip install -q --upgrade ucimlrepo pandas


In [2]:

# Step 2: Download and load the UCI Adult dataset

import pandas as pd
from ucimlrepo import fetch_ucirepo

try:
    adult = fetch_ucirepo(id=2)

    X = adult.data.features.copy()
    y = adult.data.targets.copy()

    # Normalize column names
    X.columns = X.columns.str.strip()
    y.columns = y.columns.str.strip()

    print("Dataset loaded successfully!")
    print("Number of records:", len(X))
    print("Number of input features:", X.shape[1])

    print("\nFeature names:")
    print(X.columns.tolist())

    print("\nFirst five records:")
    display(X.head())

    print("\nTarget data:")
    display(y.head())

except Exception as e:
    print("Dataset loading failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

In [2]:

# Step 3A: Clean missing values

import pandas as pd
import numpy as np

# Replace question marks and blank strings with NaN
X = X.replace(r"^\s*\?\s*$", np.nan, regex=True)
X = X.replace(r"^\s*$", np.nan, regex=True)

print("Dataset shape:", X.shape)

print("\nMissing values per column:")
print(X.isnull().sum())

print("\nTotal missing values:", X.isnull().sum().sum())


In [2]:

# Step 3B: Prepare income labels

# Standardize the target labels
y_clean = y.iloc[:, 0].astype(str).str.strip().str.rstrip(".")

# Remove rows with unexpected or missing target labels
valid_rows = y_clean.isin(["<=50K", ">50K"])

X = X.loc[valid_rows].reset_index(drop=True)
y_clean = y_clean.loc[valid_rows].reset_index(drop=True)

# Convert labels to binary values
y = (y_clean == ">50K").astype(int)

print("Target preparation complete!")

print("\nIncome class counts:")
print(y.value_counts().rename(index={
    0: "<=50K",
    1: ">50K"
}))

print("\nIncome class proportions:")
print(y.value_counts(normalize=True).round(3))


In [2]:

# Step 3C: Identify feature types

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nFinal dataset dimensions:")
print("X:", X.shape)
print("y:", y.shape)


In [2]:

# Step 4A: Split data and create a machine learning pipeline

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Identify feature types
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Process numerical features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Process categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Create complete model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Model pipeline created successfully!")


In [2]:

# Step 4B: Train the model

model.fit(X_train, y_train)

print("Model training completed successfully!")


In [2]:

# Step 4C: Evaluate the trained model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(
    precision_score(y_test, y_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, y_pred, zero_division=0), 4
))
print("F1 Score:", round(
    f1_score(y_test, y_pred, zero_division=0), 4
))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [2]:

# Step 5A: Prepare transformed data for SHAP

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Extract the fitted preprocessing component
preprocessor_fitted = model.named_steps["preprocessor"]

# Transform a manageable sample of test data
X_shap_raw = X_test.sample(
    n=min(200, len(X_test)),
    random_state=42
)

X_shap_transformed = preprocessor_fitted.transform(X_shap_raw)

# Convert sparse matrices to dense arrays when necessary
if hasattr(X_shap_transformed, "toarray"):
    X_shap_transformed = X_shap_transformed.toarray()

# Get the encoded feature names
feature_names = preprocessor_fitted.get_feature_names_out()

X_shap_df = pd.DataFrame(
    X_shap_transformed,
    columns=feature_names,
    index=X_shap_raw.index
)

print("SHAP data prepared successfully!")
print("Samples:", X_shap_df.shape[0])
print("Encoded features:", X_shap_df.shape[1])


In [2]:

# Step 5B: Calculate SHAP explanations

rf_model = model.named_steps["classifier"]

explainer = shap.TreeExplainer(rf_model)

shap_values = explainer(
    X_shap_df,
    check_additivity=False
)

print("SHAP values calculated successfully!")


In [2]:

# Step 5C: Global feature importance

shap.plots.bar(
    shap_values[:, :, 1],
    max_display=15
)


In [2]:

# Step 5D: SHAP beeswarm plot

shap.plots.beeswarm(
    shap_values[:, :, 1],
    max_display=15
)


In [2]:

# Step 5E: Explain one prediction

sample_index = 0

shap.plots.waterfall(
    shap_values[sample_index, :, 1],
    max_display=15
)


In [2]:

# Step 6A: Prepare demographic audit data

import pandas as pd
import numpy as np

audit_df = X_test.copy().reset_index(drop=True)

audit_df["actual_income"] = np.asarray(y_test)
audit_df["predicted_income"] = np.asarray(y_pred)
audit_df["predicted_probability"] = np.asarray(y_prob)

print("Fairness audit dataset created!")
print(audit_df[
    ["sex", "race", "actual_income",
     "predicted_income", "predicted_probability"]
].head())


In [2]:

# Step 6B: Compare predicted positive rates by group

def group_selection_rates(data, group_column):
    report = data.groupby(group_column, dropna=False).agg(
        sample_count=("predicted_income", "size"),
        actual_positive_rate=("actual_income", "mean"),
        predicted_positive_rate=("predicted_income", "mean"),
        mean_predicted_probability=("predicted_probability", "mean")
    )

    return report.round(4)

print("Fairness audit by sex:")
display(group_selection_rates(audit_df, "sex"))

print("Fairness audit by race:")
display(group_selection_rates(audit_df, "race"))


In [2]:

# Step 6C: Calculate selection-rate ratios

def selection_rate_ratio(data, group_column):
    rates = data.groupby(group_column)["predicted_income"].mean()

    if len(rates) < 2 or rates.max() == 0:
        return None, rates

    return rates.min() / rates.max(), rates

for column in ["sex", "race"]:
    ratio, rates = selection_rate_ratio(audit_df, column)

    print(f"\nGroup: {column}")
    print("Selection rates:")
    print(rates.round(4))

    if ratio is None:
        print("Ratio undefined: insufficient groups or no positive predictions.")
    else:
        print("Minimum-to-maximum selection-rate ratio:",
              round(float(ratio), 4))


In [2]:

# Step 6D: Group-wise classification error analysis

def group_error_report(data, group_column):
    rows = []

    for group, subset in data.groupby(group_column, dropna=False):
        actual = subset["actual_income"].astype(int)
        predicted = subset["predicted_income"].astype(int)

        tp = ((actual == 1) & (predicted == 1)).sum()
        fp = ((actual == 0) & (predicted == 1)).sum()
        fn = ((actual == 1) & (predicted == 0)).sum()
        tn = ((actual == 0) & (predicted == 0)).sum()

        rows.append({
            "group": group,
            "sample_count": len(subset),
            "true_positive_rate": tp / (tp + fn) if tp + fn else np.nan,
            "false_positive_rate": fp / (fp + tn) if fp + tn else np.nan,
            "false_negative_rate": fn / (fn + tp) if fn + tp else np.nan
        })

    return pd.DataFrame(rows).round(4)

print("Error rates by sex:")
display(group_error_report(audit_df, "sex"))

print("Error rates by race:")
display(group_error_report(audit_df, "race"))


In [2]:

# Step 6E: Visualize predicted positive rates

import matplotlib.pyplot as plt

for column in ["sex", "race"]:
    rates = audit_df.groupby(column)["predicted_income"].mean()

    ax = rates.plot(
        kind="bar",
        figsize=(7, 4),
        ylim=(0, 1),
        title=f"Predicted >50K Rate by {column}"
    )

    ax.set_xlabel(column)
    ax.set_ylabel("Predicted positive rate")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [2]:

# Step 7A: Create an AI governance audit log

import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone
from pathlib import Path

# Unique identifier for this experimental model version
MODEL_VERSION = "adult-rf-v1"

# Copy the existing fairness audit dataset
audit_log = audit_df.copy()

# Add metadata
audit_log.insert(
    0,
    "record_id",
    [f"REC-{i+1:05d}" for i in range(len(audit_log))]
)

audit_log["model_version"] = MODEL_VERSION
audit_log["logged_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Label the target and prediction clearly
audit_log["target_definition"] = "Income > $50K"
audit_log["explanation_method"] = "SHAP TreeExplainer"

# Save the log as a CSV file
output_path = Path("ai_governance_audit_log.csv")
audit_log.to_csv(output_path, index=False)

print("Audit log created successfully!")
print("Number of records:", len(audit_log))
print("Saved to:", output_path.resolve())

display(audit_log.head())


In [2]:

# Step 7B: Summarize individual SHAP explanations

# SHAP values for the positive-income class
positive_class_shap = shap_values[:, :, 1]

shap_array = np.asarray(positive_class_shap.values)
feature_names = list(X_shap_df.columns)

explanation_records = []

for i in range(len(X_shap_df)):
    contributions = shap_array[i]

    # Get the five features with the largest absolute contributions
    top_indices = np.argsort(np.abs(contributions))[-5:][::-1]

    top_features = [
        {
            "feature": feature_names[j],
            "shap_value": round(float(contributions[j]), 6)
        }
        for j in top_indices
    ]

    explanation_records.append({
        "record_id": f"SHAP-{i+1:05d}",
        "model_version": MODEL_VERSION,
        "predicted_income": int(
            model.predict(X_shap_raw.iloc[[i]])[0]
        ),
        "top_5_feature_contributions": json.dumps(top_features)
    })

explanation_df = pd.DataFrame(explanation_records)

explanation_df.to_csv(
    "shap_explanation_audit.csv",
    index=False
)

print("SHAP explanation summaries saved!")
display(explanation_df.head())


In [2]:

# Step 7C: Generate a governance summary

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

governance_summary = {
    "experiment": "Explainable AI for Responsible AI Governance",
    "dataset": "UCI Adult Census Income",
    "model": "Random Forest",
    "model_version": MODEL_VERSION,
    "target": "Income > $50K",
    "test_samples": int(len(y_test)),
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
    "explainability_method": "SHAP TreeExplainer",
    "fairness_groups_audited": ["sex", "race"],
    "audit_artifacts": [
        "ai_governance_audit_log.csv",
        "shap_explanation_audit.csv"
    ],
    "limitations": [
        "Historical dataset",
        "Observational associations are not causal explanations",
        "Fairness metrics do not prove absence of discrimination",
        "Audit log is a research prototype, not tamper-proof"
    ]
}

with open("governance_summary.json", "w") as f:
    json.dump(governance_summary, f, indent=4)

print(json.dumps(governance_summary, indent=4))


In [2]:

# Step 7D: Download audit artifacts from Google Colab

from google.colab import files

files.download("ai_governance_audit_log.csv")
files.download("shap_explanation_audit.csv")
files.download("governance_summary.json")

In [2]:

# Step 8A: Consolidated Responsible AI Governance Report

import pandas as pd
import numpy as np
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Model performance
performance = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(
        precision_score(y_test, y_pred, zero_division=0)
    ),
    "recall": float(
        recall_score(y_test, y_pred, zero_division=0)
    ),
    "f1_score": float(
        f1_score(y_test, y_pred, zero_division=0)
    ),
    "roc_auc": float(roc_auc_score(y_test, y_prob))
}

# 2. SHAP global importance
shap_class = shap_values[:, :, 1]

shap_importance = pd.DataFrame({
    "feature": X_shap_df.columns,
    "mean_absolute_shap": np.abs(
        np.asarray(shap_class.values)
    ).mean(axis=0)
}).sort_values(
    "mean_absolute_shap",
    ascending=False
)

# 3. Fairness metrics by group
def summarize_fairness(data, group_column):
    grouped = data.groupby(group_column, dropna=False)

    result = grouped.agg(
        sample_count=("predicted_income", "size"),
        actual_positive_rate=("actual_income", "mean"),
        predicted_positive_rate=("predicted_income", "mean")
    ).reset_index()

    result["group_attribute"] = group_column
    return result

fairness_results = pd.concat([
    summarize_fairness(audit_df, "sex"),
    summarize_fairness(audit_df, "race")
], ignore_index=True)

# 4. Assemble the report
report = {
    "project": "Explainable AI for Responsible AI Governance",
    "dataset": "UCI Adult Census Income",
    "model": "Random Forest",
    "target": "Annual income > $50K",
    "test_samples": int(len(y_test)),
    "performance_metrics": performance,
    "top_10_shap_features": shap_importance.head(10).to_dict(
        orient="records"
    ),
    "fairness_results": fairness_results.to_dict(
        orient="records"
    ),
    "governance_limitations": [
        "SHAP explanations describe model behavior, not causation.",
        "Group rate differences do not alone establish discrimination.",
        "Historical data may encode existing social inequalities.",
        "Further validation and human oversight are required."
    ]
}

with open("responsible_ai_governance_report.json", "w") as file:
    json.dump(report, file, indent=4)

# Export readable tables
shap_importance.to_csv("global_shap_importance.csv", index=False)
fairness_results.to_csv("fairness_evaluation.csv", index=False)

print("Governance report generated successfully!")
print("\nPerformance metrics:")
print(pd.Series(performance).round(4))

print("\nTop 10 SHAP features:")
display(shap_importance.head(10))

print("\nFairness evaluation:")
display(fairness_results)

print("\nSaved:")
print("- responsible_ai_governance_report.json")
print("- global_shap_importance.csv")
print("- fairness_evaluation.csv")


In [2]:

# Step 8B: Plot group outcome rates

import matplotlib.pyplot as plt

for attribute in ["sex", "race"]:
    rates = audit_df.groupby(attribute)["predicted_income"].mean()

    plt.figure(figsize=(8, 4))
    rates.plot(kind="bar")
    plt.title(f"Predicted Income > $50K by {attribute}")
    plt.xlabel(attribute)
    plt.ylabel("Predicted positive rate")
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(
        f"fairness_{attribute}.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


In [2]:

from google.colab import files

files.download("responsible_ai_governance_report.json")
files.download("global_shap_importance.csv")
files.download("fairness_evaluation.csv")
files.download("fairness_sex.png")
files.download("fairness_race.png")